# Experiment 08: K-Means Clustering and Elbow Method


### 1. Import Libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

print("Libraries imported successfully.")


### 2. Load and Explore the Dataset


In [ ]:
iris = load_iris(as_frame=True)
X = iris.data.copy()

print("Dataset shape (samples, features):", X.shape)
print("
First five observations:")
display(X.head())

print("
Descriptive statistics:")
display(X.describe().round(2))

print("
Missing values:", int(X.isnull().sum().sum()))


### 3. Standardize Features


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Standardized data shape:", X_scaled.shape)
print("Feature means after scaling (approx 0):", np.round(X_scaled.mean(axis=0), 3))
print("Feature standard deviations (approx 1):", np.round(X_scaled.std(axis=0), 3))


### 4. Apply K-Means for Different Values of K (Inertia / WCSS)


In [ ]:
k_values = range(1, 11)
inertias = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

elbow_data = pd.DataFrame({
    "Number of clusters (K)": list(k_values),
    "Inertia (WCSS)": inertias
})
display(elbow_data.round(3))


### 5. Plot the Elbow Curve


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(list(k_values), inertias, marker="o")
plt.xticks(list(k_values))
plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia (WCSS)")
plt.title("Elbow Method for K-Means on the Iris Dataset")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


### 6. Fit Final Model (K=3) and Inspect Cluster Sizes


In [ ]:
optimal_k = 3

final_kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
cluster_labels = final_kmeans.fit_predict(X_scaled)

clustered_data = X.copy()
clustered_data["Cluster"] = cluster_labels

print(f"Selected number of clusters: K = {optimal_k}")
print("
Cluster sizes:")
print(clustered_data["Cluster"].value_counts().sort_index())
print("
First five rows with cluster assignments:")
display(clustered_data.head())


### 7. Visualize Clusters using PCA


In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    X_pca[:, 0], X_pca[:, 1],
    c=cluster_labels, cmap="viridis",
    s=55, edgecolor="black", linewidth=0.3
)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title(f"K-Means Clusters on Iris Dataset (K={optimal_k}, PCA view)")
plt.legend(*scatter.legend_elements(), title="Cluster")
plt.grid(True, linestyle="--", alpha=0.25)
plt.tight_layout()
plt.show()

print("Variance explained by first two principal components:",
      f"{pca.explained_variance_ratio_.sum():.2%}")


### 8. Examine Cluster Centers (Original Units)


In [ ]:
centers_original_units = scaler.inverse_transform(final_kmeans.cluster_centers_)
centers_df = pd.DataFrame(centers_original_units, columns=X.columns)
centers_df.index.name = "Cluster"
display(centers_df.round(2))


### 9. Silhouette Score Analysis


In [ ]:
from sklearn.metrics import silhouette_score

silhouette_rows = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_k = km.fit_predict(X_scaled)
    silhouette_rows.append({
        "K": k,
        "Silhouette score": silhouette_score(X_scaled, labels_k)
    })

silhouette_df = pd.DataFrame(silhouette_rows)
display(silhouette_df.round(4))

plt.figure(figsize=(8, 5))
plt.plot(silhouette_df["K"], silhouette_df["Silhouette score"], marker="o")
plt.xticks(range(2, 11))
plt.xlabel("Number of clusters (K)")
plt.ylabel("Silhouette score")
plt.title("Silhouette Score by Number of Clusters")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()
